# Session 6 — English execution copy
Adapted from Dr. Loveleen Gaur's supplied notebook. Public teaching data and synthetic experiments; NOT company mining records.

The original file is retained locally, not republished. Original SHA-256: `cf1922e15082823970335b6060ad384fdd74092dbfa0ba85ecfacaa0151227cb`.

English version of Dr. Loveleen Gaur’s Session 6 notebook. AI assistance is declared in control/provenance.md.

In [1]:
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
import matplotlib
matplotlib.use('Agg')


## Adaptation notes
The original computations are retained. Bilingual print statements and plots are removed from this English copy, then replaced with English summaries and a figure. The seed experiment varies both the split and estimator seed unless noted. The last section implements all five Your turn changes.

In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GroupKFold, KFold, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
NAVY, TEAL, AMBER, RED = ('#0F2233', '#14897F', '#F2A33A', '#D9534F')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})



## Lab 1: seed variability

In [3]:
X, y = load_breast_cancer(return_X_y=True)
accuracies = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = (scaler.transform(Xtr), scaler.transform(Xte))
    model = MLPClassifier(hidden_layer_sizes=(16,), max_iter=300, random_state=seed)
    model.fit(Xtr_s, ytr)
    accuracies.append(model.score(Xte_s, yte) * 100)
accuracies = np.array(accuracies)

seed_width16 = accuracies.copy()
print('Width 16: mean, sample SD, range:', accuracies.mean(), accuracies.std(ddof=1), np.ptp(accuracies))

Width 16: mean, sample SD, range: 97.19298245614034 1.0146706621921602 3.5087719298245617


## Lab 1 extension: split versus initialization

In [4]:
def run(split_seed, init_seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=split_seed, stratify=y)
    sc = StandardScaler().fit(Xtr)
    m = MLPClassifier(hidden_layer_sizes=(16,), max_iter=300, random_state=init_seed)
    return m.fit(sc.transform(Xtr), ytr).score(sc.transform(Xte), yte) * 100
only_split = np.array([run(s, 0) for s in range(20)])
only_init = np.array([run(0, s) for s in range(20)])

print('Split-only SD:', only_split.std(ddof=1)); print('Initialization-only SD:', only_init.std(ddof=1))

Split-only SD: 1.0128952091046075
Initialization-only SD: 0.7706976931701384


## Lab 2a: scaling leakage

In [5]:
def evaluate(leaky, seed=0):
    if leaky:
        Xs = StandardScaler().fit_transform(X)
        Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=0.3, random_state=seed, stratify=y)
    else:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        sc = StandardScaler().fit(Xtr)
        Xtr, Xte = (sc.transform(Xtr), sc.transform(Xte))
    return LogisticRegression(max_iter=2000).fit(Xtr, ytr).score(Xte, yte) * 100
leaky_a = np.mean([evaluate(True, s) for s in range(30)])
clean_a = np.mean([evaluate(False, s) for s in range(30)])

print('Scaling: leaky accuracy, clean accuracy:', leaky_a, clean_a)

Scaling: leaky accuracy, clean accuracy: 97.60233918128652 97.56335282651071


## Lab 2b: feature-selection leakage

In [6]:
rng = np.random.default_rng(42)
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky_b = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=cv).mean() * 100
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
clean_b = cross_val_score(pipe, X_noise, y_noise, cv=cv).mean() * 100

print('Feature selection k20: leaky, clean:', leaky_b, clean_b)

Feature selection k20: leaky, clean: 90.0 45.0


## Lab 3: grouped independence

In [7]:
rng = np.random.default_rng(7)
n_patients, scans_each, n_feat = (80, 10, 30)
diagnosis = rng.integers(0, 2, n_patients)
fingerprint = rng.normal(0, 3.0, size=(n_patients, n_feat))
disease_dir = rng.normal(0, 1, n_feat)
disease_dir /= np.linalg.norm(disease_dir)
Xp, yp, groups = ([], [], [])
for p in range(n_patients):
    for _ in range(scans_each):
        scan = fingerprint[p] + 0.6 * diagnosis[p] * disease_dir + rng.normal(0, 1, n_feat)
        Xp.append(scan)
        yp.append(diagnosis[p])
        groups.append(p)
Xp, yp, groups = (np.array(Xp), np.array(yp), np.array(groups))
rf = RandomForestClassifier(n_estimators=200, random_state=0)
by_scan = cross_val_score(rf, Xp, yp, cv=KFold(5, shuffle=True, random_state=0)).mean() * 100
by_patient = cross_val_score(rf, Xp, yp, cv=GroupKFold(5), groups=groups).mean() * 100

patient_gap_baseline = (by_scan, by_patient)
print('Fingerprint strength 3.0: by scan, by patient:', by_scan, by_patient)

Fingerprint strength 3.0: by scan, by patient: 99.25 54.25000000000001


## Lab 4a: repeated peeking

In [8]:
def run_experiment(peek, n_final=2000, checks=10, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.random(n_final) < 0.1
    b = rng.random(n_final) < 0.1
    if not peek:
        return stats.ttest_ind(a, b).pvalue < 0.05
    for k in range(1, checks + 1):
        m = int(n_final * k / checks)
        if stats.ttest_ind(a[:m], b[:m]).pvalue < 0.05:
            return True
    return False
N = 2000
honest_fp = np.mean([run_experiment(False, seed=s) for s in range(N)]) * 100
peek_fp = np.mean([run_experiment(True, seed=s) for s in range(N)]) * 100

peek10 = peek_fp
print('No peeking and 10 checks: false-positive percentages:', honest_fp, peek_fp)

No peeking and 10 checks: false-positive percentages: 4.65 18.6


## Lab 4b: multiple comparisons

In [9]:
rng = np.random.default_rng(3)
studies, subgroups = (2000, 20)
found_any, found_any_holm = (0, 0)
for _ in range(studies):
    p = np.array([stats.ttest_ind(rng.normal(size=50), rng.normal(size=50)).pvalue for _ in range(subgroups)])
    found_any += (p < 0.05).any()
    order = np.sort(p)
    thresholds = 0.05 / (subgroups - np.arange(subgroups))
    found_any_holm += order[0] < thresholds[0]

print('Uncorrected and Holm family false-positive rates:', found_any/studies, found_any_holm/studies)

Uncorrected and Holm family false-positive rates: 0.6565 0.0465


## Lab 5a: significance versus effect size

In [10]:
rng = np.random.default_rng(11)
n = 5000000
conv_a = rng.binomial(n, 0.1)
conv_b = rng.binomial(n, 0.1005)
pa, pb = (conv_a / n, conv_b / n)
diff = pb - pa
se = np.sqrt(pa * (1 - pa) / n + pb * (1 - pb) / n)
z = diff / se
p = 2 * (1 - stats.norm.cdf(abs(z)))
lo, hi = (diff - 1.96 * se, diff + 1.96 * se)

print('Large sample: difference, p-value, CI:', diff, p, lo, hi)

Large sample: difference, p-value, CI: 0.0008225999999999928 1.4949041507561489e-05 0.0004501969222015628 0.0011950030777984227


## Lab 5b: sample size and uncertainty

In [11]:
rng = np.random.default_rng(5)
control = rng.normal(70, 10, 20)
trained = rng.normal(76, 10, 20)
t = stats.ttest_ind(trained, control)
d = (trained.mean() - control.mean()) / np.sqrt((trained.var(ddof=1) + control.var(ddof=1)) / 2)
diff = trained.mean() - control.mean()
se = np.sqrt(trained.var(ddof=1) / 20 + control.var(ddof=1) / 20)
ci = stats.t.interval(0.95, df=38, loc=diff, scale=se)
from math import ceil
z_a, z_b = (stats.norm.ppf(0.975), stats.norm.ppf(0.8))
n_needed = ceil(2 * ((z_a + z_b) / 0.6) ** 2)

small_n_effect = (diff, d, float(t.pvalue), tuple(float(v) for v in ci))
print('n20: mean difference, Cohen d, p, CI:', small_n_effect)
print('Approximate participants per group for the stated power target:', n_needed)

n20: mean difference, Cohen d, p, CI: (np.float64(3.4309208077508373), np.float64(0.4109153385928523), 0.20162544225077228, (-1.9141494434154147, 8.77599105891709))
Approximate participants per group for the stated power target: 44


## Bootstrap interval

In [12]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
correct = (clf.predict(Xte) == yte).astype(float)
rng = np.random.default_rng(0)
boot = [rng.choice(correct, size=len(correct), replace=True).mean() * 100 for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])

print('Bootstrap accuracy percent and interval:', correct.mean()*100, lo, hi)

Bootstrap accuracy percent and interval: 95.90643274853801 92.98245614035088 98.83040935672514


## Advanced comparison: corrected versus naive test

In [13]:
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=0)
model_a = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model_b = RandomForestClassifier(n_estimators=200, random_state=0)
acc_a = cross_val_score(model_a, X, y, cv=cv) * 100
acc_b = cross_val_score(model_b, X, y, cv=cv) * 100
d = acc_a - acc_b
k = len(d)
n_test = len(y) // 10
n_train = len(y) - n_test
t_naive = d.mean() / (d.std(ddof=1) / np.sqrt(k))
p_naive = 2 * stats.t.sf(abs(t_naive), df=k - 1)
t_corr = d.mean() / np.sqrt((1 / k + n_test / n_train) * d.var(ddof=1))
p_corr = 2 * stats.t.sf(abs(t_corr), df=k - 1)

print('Paired mean gap, naive p, corrected p:', d.mean(), p_naive, p_corr)

Paired mean gap, naive p, corrected p: 1.6359649122807007 9.581627358829224e-09 0.07243857548146883


## Reproducibility environment

In [14]:
import random, os, sys, platform, sklearn, scipy

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
set_all_seeds(42)

print('Environment:', sys.version.split()[0], np.__version__, sklearn.__version__, scipy.__version__)

Environment: 3.12.14 2.2.6 1.6.1 1.18.1


## Your turn 1 — width 64
Keep the original 20 split/init seeds; compare dispersion without assuming that a larger network must be more stable.

In [15]:
X, y = load_breast_cancer(return_X_y=True)
accuracies = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = (scaler.transform(Xtr), scaler.transform(Xte))
    model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=seed)
    model.fit(Xtr_s, ytr)
    accuracies.append(model.score(Xte_s, yte) * 100)
accuracies = np.array(accuracies)

seed_width64=accuracies.copy()
print('Width16 sample SD:', seed_width16.std(ddof=1)); print('Width64 sample SD:', seed_width64.std(ddof=1))


Width16 sample SD: 1.0146706621921602
Width64 sample SD: 0.919273810289908


## Your turn 2 — k=5 and k=100
Selection on all rows is the deliberately leaky branch. The clean branch selects inside each fold.

In [16]:
leakage_variants=[]
for k_features in [5,20,100]:
    rng=np.random.default_rng(42)
    X_noise=rng.normal(size=(100,5000));y_noise=rng.integers(0,2,size=100)
    cv_local=StratifiedKFold(5,shuffle=True,random_state=0)
    selected=SelectKBest(f_classif,k=k_features).fit_transform(X_noise,y_noise)
    leaked=cross_val_score(LogisticRegression(max_iter=1000),selected,y_noise,cv=cv_local).mean()*100
    safe=make_pipeline(SelectKBest(f_classif,k=k_features),LogisticRegression(max_iter=1000))
    honest=cross_val_score(safe,X_noise,y_noise,cv=cv_local).mean()*100
    leakage_variants.append({'k':k_features,'leaky_accuracy_pct':float(leaked),'clean_accuracy_pct':float(honest)})
print(leakage_variants)


[{'k': 5, 'leaky_accuracy_pct': 77.99999999999999, 'clean_accuracy_pct': 49.00000000000001}, {'k': 20, 'leaky_accuracy_pct': 90.0, 'clean_accuracy_pct': 45.0}, {'k': 100, 'leaky_accuracy_pct': 99.0, 'clean_accuracy_pct': 48.0}]


## Your turn 3 — fingerprint strength 0.5
Record the change in the scan-versus-patient gap. This is an independence demonstration, not a mining result.

In [17]:
rng = np.random.default_rng(7)
n_patients, scans_each, n_feat = (80, 10, 30)
diagnosis = rng.integers(0, 2, n_patients)
fingerprint = rng.normal(0, 0.5, size=(n_patients, n_feat))
disease_dir = rng.normal(0, 1, n_feat)
disease_dir /= np.linalg.norm(disease_dir)
Xp, yp, groups = ([], [], [])
for p in range(n_patients):
    for _ in range(scans_each):
        scan = fingerprint[p] + 0.6 * diagnosis[p] * disease_dir + rng.normal(0, 1, n_feat)
        Xp.append(scan)
        yp.append(diagnosis[p])
        groups.append(p)
Xp, yp, groups = (np.array(Xp), np.array(yp), np.array(groups))
rf = RandomForestClassifier(n_estimators=200, random_state=0)
by_scan = cross_val_score(rf, Xp, yp, cv=KFold(5, shuffle=True, random_state=0)).mean() * 100
by_patient = cross_val_score(rf, Xp, yp, cv=GroupKFold(5), groups=groups).mean() * 100

patient_gap_reduced=(by_scan,by_patient)
print('Strength3.0:', patient_gap_baseline); print('Strength0.5:',patient_gap_reduced)


Strength3.0: (np.float64(99.25), np.float64(54.25000000000001))
Strength0.5: (np.float64(56.25), np.float64(48.75))


## Your turn 4 — 50 checks
Use the same 2,000 simulation seeds and final sample size.

In [18]:
peek50=float(np.mean([run_experiment(True,checks=50,seed=s) for s in range(2000)])*100)
print('False-positive percentages, 10 versus 50 checks:',peek10,peek50)


False-positive percentages, 10 versus 50 checks: 18.6 31.5


## Your turn 5 — 70 participants per group
Update the standard-error denominators and degrees of freedom as well as sample size. The generating mean difference remains six points; the observed estimate can change.

In [19]:
rng = np.random.default_rng(5)
control = rng.normal(70, 10, 70)
trained = rng.normal(76, 10, 70)
t = stats.ttest_ind(trained, control)
d = (trained.mean() - control.mean()) / np.sqrt((trained.var(ddof=1) + control.var(ddof=1)) / 2)
diff = trained.mean() - control.mean()
se = np.sqrt(trained.var(ddof=1) / 70 + control.var(ddof=1) / 70)
ci = stats.t.interval(0.95, df=138, loc=diff, scale=se)
from math import ceil
z_a, z_b = (stats.norm.ppf(0.975), stats.norm.ppf(0.8))
n_needed = ceil(2 * ((z_a + z_b) / 0.6) ** 2)

large_n_effect=(float(diff),float(d),float(t.pvalue),tuple(float(v) for v in ci))
print('n20 effect:',small_n_effect);print('n70 effect:',large_n_effect)


n20 effect: (np.float64(3.4309208077508373), np.float64(0.4109153385928523), 0.20162544225077228, (-1.9141494434154147, 8.77599105891709))
n70 effect: (6.397802568933798, 0.7148614297267684, 4.246333000292e-05, (3.4065830180406813, 9.389022119826915))


In [20]:
from pathlib import Path
import json
results={'data_origin':'PUBLIC_TEACHING_DATA_AND_SYNTHETIC_EXPERIMENTS','seed_width16_sd':float(seed_width16.std(ddof=1)),'seed_width64_sd':float(seed_width64.std(ddof=1)),'leakage':leakage_variants,'patient_strength3':patient_gap_baseline,'patient_strength0_5':patient_gap_reduced,'peek10_percent':float(peek10),'peek50_percent':float(peek50),'n20_effect':small_n_effect,'n70_effect':large_n_effect}
Path('results.json').write_text(json.dumps(results,indent=2)+'\n',encoding='utf-8')
plt.figure(figsize=(7,4));plt.bar(['No peeking','10 checks','50 checks'],[honest_fp,peek10,peek50],color=['#237a67','#d0923a','#a74e42']);plt.ylabel('False-positive rate (%)');plt.title('Session 6: repeated peeking — synthetic experiments');plt.tight_layout();plt.savefig('peeking_comparison.png',dpi=150);plt.close()
print('Results saved.')


Results saved. Reflection and exit-ticket statement remain for the student to write.
